# Dataset & Dataloader Testing

Tests for `DatasetCatCon`, dataloader throughput, and DDP sampler coverage.

Sections:
1. Imports & path setup
2. Config loading (Hydra compose)
3. Dataset instantiation & single-sample inspection
4. Dataloader throughput benchmark
5. DistributedSampler coverage analysis (static — no real DDP needed)
6. Multi-process DDP simulation via `torch.multiprocessing.spawn`

In [1]:
import os, sys, time

# Add src/ to path so the package is importable without installing
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, DistributedSampler

## 1 · Config loading

In [2]:
# Hydra compose without starting a CLI process.
# Change config_name to the model config you want to test.
from hydra import compose, initialize_config_dir

CONFIG_NAME = "atlas_tracks_saint"   # ← change as needed

with initialize_config_dir(
    config_dir=os.path.abspath('../configs'), version_base=None
):
    cfg = compose(config_name=CONFIG_NAME)

print(OmegaConf.to_yaml(cfg.dataloader))
print("global_object :", cfg.global_object)
print("constituents  :", list(cfg.constituent_objects))

batch_size: 1024
num_workers: 4
prefetch_factor: 2
pin_memory: true

global_object : jets
constituents  : ['tracks']


## 2 · Dataset instantiation & single-sample inspection

`DatasetCatCon` opens the HDF5 file lazily (one handle per worker), so the
first `__getitem__` call opens the file.

In [3]:
from fm4tag.datasets import DatasetCatCon

def load_dicts(cfg):
    """Load norm_dict / class_dict from paths in cfg (or return None)."""
    norm  = OmegaConf.to_container(OmegaConf.load(cfg.norm_dict),  resolve=True) if cfg.get("norm_dict")  else None
    klass = OmegaConf.to_container(OmegaConf.load(cfg.class_dict), resolve=True) if cfg.get("class_dict") else None
    return norm, klass

norm_dict, class_dict = load_dicts(cfg)

dataset = DatasetCatCon(
    file_path=cfg.pretrain_file,
    variables=cfg.variables,
    global_object=cfg.global_object,
    constituent_objects=list(cfg.constituent_objects),
    norm_dict=norm_dict,
    class_dict=class_dict,
)
print(f"\nDataset length : {len(dataset):,}")


DatasetCatCon: /storage3/DSIP/rriva/datasets/transforming-jet-flavor/data_parallel_preprocess_251112_112137/pretrain/output/pp_output_pretrain.h5
  samples : 500000

Dataset length : 500,000


In [4]:
# Inspect a single sample
sample = dataset[0]
print("label :", sample["label"])
print("global:", sample["global"].shape, sample["global"].dtype)
for name, obj in sample["constituents"].items():
    print(f"  {name}")
    print(f"    categorical : {obj['categorical'].shape}  {obj['categorical'].dtype}")
    print(f"    continuous  : {obj['continuous'].shape}   {obj['continuous'].dtype}")
    print(f"    valid       : {obj['valid'].shape}")
    n_valid = obj['valid'].sum().item()
    print(f"    # valid constituents in this sample: {n_valid}")

label : tensor(1)
global: torch.Size([2]) torch.float32
  tracks
    categorical : torch.Size([40, 9])  torch.uint8
    continuous  : torch.Size([40, 10])   torch.float32
    valid       : torch.Size([40])
    # valid constituents in this sample: 5


In [8]:
# Quick sanity: continuous features should be normalised ≈ N(0,1)
# (check a few batches to get a better mean/std estimate)
from fm4tag.datasets import cat_con_collate_fn

quick_loader = DataLoader(
    dataset, batch_size=2048, shuffle=True, num_workers=0,
    collate_fn=cat_con_collate_fn,
)
batch = next(iter(quick_loader))
for name, obj in batch["constituents"].items():
    x = obj["continuous"]
    mask = obj["valid"]                      # (B, N)
    flat = x[mask]                           # (M, F_con)
    print(f"{name}  continuous — mean: {flat.mean(0).numpy().round(3)}")
    print(f"{' '*len(name)}             std : {flat.std(0).numpy().round(3)}")

tracks  continuous — mean: [ 0.005 -0.013  0.009  0.009  0.001  0.012 -0.002 -0.009 -0.006 -0.009]
                   std : [0.989 0.984 1.002 0.997 0.998 1.041 1.012 0.974 0.99  0.921]


## 3 · Dataloader throughput benchmark

Measures samples/second for different `num_workers` values.
Adjust `N_BATCHES` and `BATCH_SIZE` to match your typical training setup.

In [9]:
BATCH_SIZE = int(cfg.pretrain_dataloader.get("batch_size", 1024))
N_BATCHES  = 40   # batches to time after the warmup

def benchmark(dataset, batch_size, num_workers, n_batches=N_BATCHES, n_warmup=3):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        collate_fn=cat_con_collate_fn,
        prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=False,
        pin_memory=True,
    )
    it = iter(loader)
    for _ in range(n_warmup):      # warmup: fill prefetch buffers
        next(it)

    t0 = time.perf_counter()
    n_samples = 0
    for i, batch in enumerate(it):
        n_samples += batch["label"].shape[0]
        if i + 1 >= n_batches:
            break
    elapsed = time.perf_counter() - t0

    return {
        "samples/s": n_samples / elapsed,
        "batches/s": (i + 1)  / elapsed,
        "ms/batch" : elapsed  / (i + 1) * 1e3,
    }

print(f"batch_size = {BATCH_SIZE}  |  timing {N_BATCHES} batches\n")
print(f"{'workers':>8}  {'samples/s':>12}  {'batches/s':>10}  {'ms/batch':>9}")
print("-" * 46)
for nw in [0, 2, 4, 8]:
    r = benchmark(dataset, BATCH_SIZE, nw)
    print(f"{nw:>8}  {r['samples/s']:>12,.0f}  {r['batches/s']:>10.1f}  {r['ms/batch']:>9.1f}")

batch_size = 2048  |  timing 40 batches

 workers     samples/s   batches/s   ms/batch
----------------------------------------------
       0           732         0.4     2797.2
       2         1,547         0.8     1324.1
       4         2,954         1.4      693.3
       8         5,972         2.9      343.0


## 4 · DistributedSampler coverage analysis (static)

Verifies that with `world_size = N` processes each rank receives a
**non-overlapping, complete partition** of the dataset.

This is a pure index-math check — no actual GPU or `dist.init_process_group`
call needed.  It directly addresses the bug of *"4 GPUs but only one was
using 1/4 of the data"*: correct DDP should have each rank use its **own**
exclusive 1/4.

In [10]:
WORLD_SIZE = 4   # simulated number of processes/GPUs
EPOCH      = 0   # DistributedSampler changes order per epoch when shuffle=True

rank_indices = {}
for rank in range(WORLD_SIZE):
    sampler = DistributedSampler(
        dataset,
        num_replicas=WORLD_SIZE,
        rank=rank,
        shuffle=False,   # deterministic for verification
        drop_last=False,
    )
    sampler.set_epoch(EPOCH)
    rank_indices[rank] = list(sampler)

print(f"Dataset size : {len(dataset):,}")
print(f"World size   : {WORLD_SIZE}")
print()
for rank, idxs in rank_indices.items():
    print(f"  rank {rank}: {len(idxs):,} samples  (first 5: {idxs[:5]})")

Dataset size : 500,000
World size   : 4

  rank 0: 125,000 samples  (first 5: [0, 4, 8, 12, 16])
  rank 1: 125,000 samples  (first 5: [1, 5, 9, 13, 17])
  rank 2: 125,000 samples  (first 5: [2, 6, 10, 14, 18])
  rank 3: 125,000 samples  (first 5: [3, 7, 11, 15, 19])


In [11]:
# ── Coverage & overlap checks ───────────────────────────────────────────────
all_sets = {r: set(idxs) for r, idxs in rank_indices.items()}
all_flat  = [idx for s in all_sets.values() for idx in s]

print("=== Coverage check ===")
print(f"  Total samples across all ranks : {len(all_flat):,}")
print(f"  Unique samples                 : {len(set(all_flat)):,}")
print(f"  Dataset size                   : {len(dataset):,}")
coverage_ok = len(set(all_flat)) == len(dataset) or len(set(all_flat)) == len(dataset) - 1
print(f"  Full coverage ✓" if coverage_ok else f"  ⚠ Missing samples!")

print()
print("=== Pairwise overlap check ===")
any_overlap = False
for r1 in range(WORLD_SIZE):
    for r2 in range(r1 + 1, WORLD_SIZE):
        overlap = all_sets[r1] & all_sets[r2]
        if overlap:
            print(f"  ⚠ Ranks {r1} and {r2} share {len(overlap)} samples!")
            any_overlap = True
if not any_overlap:
    print(f"  No pairwise overlap ✓  (ranks are fully disjoint)")

=== Coverage check ===
  Total samples across all ranks : 500,000
  Unique samples                 : 500,000
  Dataset size                   : 500,000
  Full coverage ✓

=== Pairwise overlap check ===
  No pairwise overlap ✓  (ranks are fully disjoint)


In [12]:
# ── Show what epoch shuffling does to the partition ─────────────────────────
print("Checking that set_epoch changes order but not partition size:")
for epoch in [0, 1, 2]:
    sampler = DistributedSampler(dataset, num_replicas=WORLD_SIZE, rank=0, shuffle=True)
    sampler.set_epoch(epoch)
    idxs = list(sampler)
    print(f"  epoch {epoch}: {len(idxs):,} samples, first 5: {idxs[:5]}")

Checking that set_epoch changes order but not partition size:
  epoch 0: 125,000 samples, first 5: [136044, 369675, 112617, 467864, 32520]
  epoch 1: 125,000 samples, first 5: [95845, 491267, 321847, 728, 157846]
  epoch 2: 125,000 samples, first 5: [83848, 301246, 494848, 480295, 48239]


## 5 · Multi-process DDP simulation via `mp.spawn`

Spawns `WORLD_SIZE` **CPU** processes (gloo backend — no GPU needed) where
each process:
1. Initialises a distributed process group
2. Creates a `DataLoader` with `DistributedSampler`
3. Iterates one epoch and collects all sample indices
4. Stores the result in a shared list

After all processes finish we verify the same coverage & disjointness as
the static check above, but now through a real multi-process run.

In [13]:
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.utils.data import DistributedSampler, DataLoader

# We pass config kwargs rather than the dataset object itself to avoid
# pickling issues with open HDF5 file handles.
_DATASET_KWARGS = dict(
    file_path         = cfg.pretrain_file,
    variables         = cfg.variables,
    global_object     = cfg.global_object,
    constituent_objects = list(cfg.constituent_objects),
    norm_dict         = norm_dict,
    class_dict        = class_dict,
)

def _ddp_worker(rank, world_size, store_port, result_queue):
    """Runs inside each spawned process."""
    # Initialise process group with gloo (CPU, no NCCL needed)
    dist.init_process_group(
        backend    = "gloo",
        init_method= f"tcp://127.0.0.1:{store_port}",
        world_size = world_size,
        rank       = rank,
    )

    # Re-create dataset inside the worker (HDF5 file opened fresh per process)
    from fm4tag.datasets import DatasetCatCon, cat_con_collate_fn
    ds = DatasetCatCon(**_DATASET_KWARGS)

    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank, shuffle=False)
    sampler.set_epoch(0)
    loader  = DataLoader(ds, batch_size=512, sampler=sampler,
                         num_workers=0, collate_fn=cat_con_collate_fn)

    collected = []
    start = 0
    for batch in loader:
        bs = batch["label"].shape[0]
        # Reconstruct original indices from sampler ordering
        # (simplest: just count batch sizes to check coverage)
        collected.append(bs)
        start += bs

    result_queue.put((rank, sum(collected)))
    dist.destroy_process_group()

In [14]:
import random

WORLD_SIZE_MP = 4   # number of simulated processes (keep ≤ CPU cores)
PORT          = random.randint(29500, 29900)   # pick a free port

if __name__ == "__main__" or True:   # True makes this work inside Jupyter
    ctx    = mp.get_context("spawn")
    queue  = ctx.Queue()

    processes = []
    for rank in range(WORLD_SIZE_MP):
        p = ctx.Process(
            target = _ddp_worker,
            args   = (rank, WORLD_SIZE_MP, PORT, queue),
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    results = {}
    while not queue.empty():
        rank, n = queue.get()
        results[rank] = n

    print(f"Samples seen per rank (world_size={WORLD_SIZE_MP}):")
    total = 0
    for rank in sorted(results):
        print(f"  rank {rank}: {results[rank]:,} samples")
        total += results[rank]
    print(f"  Total   : {total:,}")
    print(f"  Dataset : {len(dataset):,}")
    print()
    expected = len(dataset) // WORLD_SIZE_MP + (1 if len(dataset) % WORLD_SIZE_MP else 0)
    print(f"  Expected per rank (approx): {expected:,}")
    all_equal = len(set(results.values())) <= 2   # allow ±1 from drop_last rounding
    print("  Load balanced ✓" if all_equal else "  ⚠ Unequal load — check sampler!")

Samples seen per rank (world_size=4):
  Total   : 0
  Dataset : 500,000

  Expected per rank (approx): 125,000
  Load balanced ✓


Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=144, pipe_handle=148)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/rriva/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/home/rriva/.local/share/uv/python/cpython-3.13.7-linux-x86_64-gnu/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute '_ddp_worker' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=144, pipe_handle=178)
                

### What the test catches

| Symptom | Root cause |
|---|---|
| All ranks get identical indices | `DistributedSampler` not used — each rank iterates the full dataset |
| One rank gets all data, others 0 | Process group init failed; only rank 0 ran |
| Rank 0 gets 1/4, ranks 1-3 get 0 | Data split correctly but other ranks not computing (stale or hung) |
| All ranks get 1/4 but overlapping | Wrong `rank` argument passed to `DistributedSampler` |

Lightning automatically wraps your DataLoader with `DistributedSampler` via
`use_distributed_sampler=True` (the default). **Do not** add a
`DistributedSampler` manually in `_make_dataloader` — doing both results in
double-sampling where each rank only sees 1/N² of the data.